In [1]:
import argparse
import pandas as pd
import numpy as np
from preprocess import preprocess
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,precision_recall_curve, precision_score, recall_score, average_precision_score, roc_curve, auc, confusion_matrix, mean_squared_error,classification_report
import time

import matplotlib.pyplot as plt
from keras.utils import to_categorical

train = pd.read_csv('./data/NSL-KDD/KDDTrain+.txt', sep=",", header=None)
test = pd.read_csv('./data/NSL-KDD/KDDTest+.txt', sep=",", header=None)


Using TensorFlow backend.


In [2]:
processor = preprocess()
print("数据预处理....")
df_train, df_test, train_Normal, train_R2L, train_U2R, train_Dos, train_Probe,test_Normal, test_R2L, test_U2R, test_Dos, test_Probe,train_Attack,test_Attack = processor.create_df(df_train=train, df_test=test)
# normal_df, R2L_df, R2L_df_train, R2L_df_test = processor.create_df(df_train=trainset, df_test=testset)
print("已完成数据预处理")

数据预处理....
已完成数据预处理


In [3]:
# lbl = ["Duration", "Protocol_type", "Service", "Flag", "Src_bytes", "Dst_bytes",
#            "Land", "Wrong_fragment", "Urgent", "Hot", "Num_failed_logins", "Logged_in",
#            "Num_compromised", "Root_shell", "Su_attempted", "Num_root", "Num_file_creations",
#            "Num_shells", "Num_access_files", "Num_outbound_cmds", "Is_hot_login",
#            "Is_guest_login", "Count", "Srv_count", "Serror_rate", "Srv_serror_rate",
#            "Rerror_rate", "Srv_rerror_rate", "Same_srv_rate", "Diff_srv_rate",
#            "Srv_diff_host_rate", "Dst_host_count", "Dst_host_srv_count", "Dst_host_same_srv_rate",
#            "Dst_host_diff_srv_rate", "Dst_host_same_src_port_rate", "Dst_host_srv_diff_host_rate",
#            "Dst_host_serror_rate", "Dst_host_srv_serror_rate", "Dst_host_rerror_rate",
#            "Dst_host_srv_rerror_rate", "attack_type", "Class"]


# combined_data = pd.concat([train, test])
# combined_data.columns = lbl
# combined_data.head(3)

In [4]:
train.head(3)

,Duration,Protocol_type,Service,Flag,Src_bytes,Dst_bytes,Land,Wrong_fragment,Urgent,Hot,...,Dst_host_same_srv_rate,Dst_host_diff_srv_rate,Dst_host_same_src_port_rate,Dst_host_srv_diff_host_rate,Dst_host_serror_rate,Dst_host_srv_serror_rate,Dst_host_rerror_rate,Dst_host_srv_rerror_rate,attack_type,Class
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.0,0.0,0.0,0.05,0.0,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.0,0.0,0.0,0.00,0.0,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.0,1.0,1.0,0.00,0.0,neptune,19


In [5]:
# # Contaminsation mean pollution (outliers) in data
# tmp = train.where(train['attack_cat'] == "Normal").dropna()
# contamination = round(1 - len(tmp)/len(train), 2)
# print("train contamination ", contamination)

# tmp = test.where(test['attack_cat'] == "Normal").dropna()
# print("test  contamination ", round(1 - len(tmp)/len(test),2),'\n')

# if contamination > 0.5:
#     print(f'contamination is {contamination}, which is greater than 0.5. Fixing...')
#     contamination = round(1-contamination,2)
#     print(f'contamination is now {contamination}')

In [6]:
dt={'Dos':pd.Series([len(train_Dos),len(test_Dos)],index=['Train','Test']),
   'Probe':pd.Series([len(train_Probe),len(test_Probe)],index=['Train','Test']),
   'R2L':pd.Series([len(train_R2L),len(test_R2L)],index=['Train','Test']),
   'U2R':pd.Series([len(train_U2R),len(test_U2R)],index=['Train','Test']),
   'Normal':pd.Series([len(train_Normal),len(test_Normal)],index=['Train','Test']),
   'Total_attack':pd.Series([len(train_Attack),len(test_Attack)],index=['Train','Test']),
   'Total':pd.Series([len(df_train),len(df_test)],index=['Train','Test'])}
type_df=pd.DataFrame(dt)
cols = ['Dos','Probe','R2L','U2R','Normal','Total_attack','Total']
type_df = type_df[cols]
display(type_df)


,Dos,Probe,R2L,U2R,Normal,Total_attack,Total
Train,11656,45927,995,52,67343,58630,125973
Test,2421,7460,2885,67,9711,12833,22544


In [21]:
from preprocess import preprocess

#加入生成数据的R2L类别二分类
generated_data = pd.read_csv('./output/fake_examples.csv', sep=",", header=None)
generated_data = processor.gererated_preprocess(generated_data)
generated_data.head(3)

,Duration,Protocol_type,Service,Flag,Src_bytes,Dst_bytes,Land,Wrong_fragment,Urgent,Hot,...,Dst_host_srv_count,Dst_host_same_srv_rate,Dst_host_diff_srv_rate,Dst_host_same_src_port_rate,Dst_host_srv_diff_host_rate,Dst_host_serror_rate,Dst_host_srv_serror_rate,Dst_host_rerror_rate,Dst_host_srv_rerror_rate,attack_type
0,0.007608,0.465964,0.795613,0.874634,0.019010,0.019146,0.018665,0.020190,0.019336,0.053437,...,0.043150,0.996327,0.018580,0.161868,-0.006656,0.006478,0.027007,0.003728,0.017637,1
1,0.002804,0.463421,0.809682,0.881286,0.018738,0.018817,0.019099,0.018611,0.016644,0.022528,...,0.029454,0.993562,0.018257,0.354078,0.032573,0.025412,0.041215,0.016875,0.016831,1
2,0.002676,0.457921,0.805100,0.877900,0.025986,0.018105,0.018363,0.018607,0.018725,0.018711,...,0.027088,0.991950,0.018511,0.329815,0.037050,0.026844,0.039289,0.014938,0.014417,1


In [22]:
#增加生成数据后，随机划分的全局二分类
from sklearn.model_selection import train_test_split

train_Bi = df_train.append(generated_data)
test_Bi = df_test

combined_data = pd.concat([train_Bi, test_Bi])
combined_data = combined_data.replace(4,1).replace(3,1).replace(2,1)
data_x = combined_data.drop(['attack_type'], axis=1) # droped label
data_y = combined_data.loc[:,['attack_type']]
# del combined_data # free mem
X_train, X_test, y_train, y_test = train_test_split(data_x, data_y, test_size=.20, random_state=42) # TODO


In [23]:
print(X_train.shape) 
print(X_test.shape) 
print(y_train.shape) 
print(y_test.shape)

(122813, 41)
(30704, 41)
(122813, 1)
(30704, 1)


In [24]:
#增加生成数据的非随机 单独二分类

#train_add_BiR2L = train_Normal.append(generated_data)

#train_add_BiR2L = processor.merge_df(df_train, generated_data)
#train_add_BiR2L = np.concatenate(df_train, generated_data)

train_BiR2L = train_Normal.append(train_R2L)
train_add_BiR2L = train_BiR2L.append(generated_data)
X_train_add_R2L,y_train_add_R2L = processor.split_df(train_add_BiR2L)

test_BiR2L = test_Normal.append(test_R2L)
X_test_R2L,y_test_R2L = processor.split_df(test_BiR2L)

X_train_add_R2L.shape

(73338, 41)

In [25]:
#全局二分类
df_train2 = df_train.append(generated_data)
X_train_Bi,y_train_Bi = processor.split_df(df_train2)
y_train_Bi = y_train_Bi.replace(4,1).replace(3,1).replace(2,1)

X_test_Bi,y_test_Bi = processor.split_df(df_test)
y_test_Bi = y_test_Bi.replace(4,1).replace(3,1).replace(2,1)




In [26]:
#Dos类二分类
train_BiDos = train_Normal.append(train_Dos)
X_train_Dos,y_train_Dos = processor.split_df(train_BiDos)
y_train_Dos = y_train_Dos.replace(2,1)

test_BiDos = test_Normal.append(test_Dos)
X_test_Dos,y_test_Dos = processor.split_df(test_BiDos)
y_test_Dos = y_test_Dos.replace(2,1)

In [27]:
#R2L类二分类
#train_BiR2L = train_Normal.append(train_R2L)
X_train_R2L,y_train_R2L = processor.split_df(train_BiR2L)


# test_BiR2L = test_Normal.append(test_R2L)
# X_test_R2L,y_test_R2L = processor.split_df(test_BiR2L)
X_train_R2L.shape

(68338, 41)

In [28]:
#U2R类二分类
train_BiU2R = train_Normal.append(train_U2R)
X_train_U2R,y_train_U2R = processor.split_df(train_BiU2R)
y_train_U2R = y_train_U2R.replace(4,1)

test_BiU2R = test_Normal.append(test_U2R)
X_test_U2R,y_test_U2R = processor.split_df(test_BiU2R)
y_test_U2R = y_test_U2R.replace(4,1)

In [29]:
#Probe类二分类
train_BiProbe = train_Normal.append(train_Probe)
X_train_Probe,y_train_Probe = processor.split_df(train_BiProbe)
y_train_Probe = y_train_Probe.replace(3,1)

test_BiProbe = test_Normal.append(test_Probe)
X_test_Probe,y_test_Probe = processor.split_df(test_BiProbe)
y_test_Probe = y_test_Probe.replace(3,1)

In [30]:
# X = X_train_add_R2L
# Y = y_train_add_R2L
# C = y_test_R2L
# T = X_test_R2L

# X = X_train_Bi
# Y = y_train_Bi
# C = y_test_Bi
# T = X_test_Bi

X = X_train
T = X_test
Y = y_train
C = y_test

In [31]:
Y = Y.astype(int)


In [32]:
import numpy as np
import pandas as pd
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn import svm
from sklearn.metrics import classification_report
from sklearn import metrics
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (precision_score, recall_score,f1_score, accuracy_score,mean_squared_error,mean_absolute_error)
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import Normalizer
from pandas_ml import ConfusionMatrix
# traindata = pd.read_csv('UNSW_NB15_training-set.csv', header=None)
# testdata = pd.read_csv('UNSW_NB15_testing-set.csv', header=None)
# traindata = pd.read_csv('kddtrain.csv', header=None)
# testdata = pd.read_csv('kddtest.csv', header=None)


# X = traindata.iloc[:,1:42]
# Y = traindata.iloc[:,0]
# C = testdata.iloc[:,0]
# T = testdata.iloc[:,1:42]


scaler = Normalizer().fit(X)
trainX = scaler.transform(X)

scaler = Normalizer().fit(T)
testT = scaler.transform(T)


traindata = np.array(trainX)
trainlabel = np.array(Y)

testdata = np.array(testT)
testlabel = np.array(C)

testlabel = testlabel.flatten()
testlabel = testlabel.astype(int)

model = LogisticRegression()
model.fit(traindata, trainlabel)


# make predictions
expected = testlabel

predicted = model.predict(testdata)

#predicted = predicted.reshape(len(predicted),1)
print(expected.shape)
print(predicted.shape)

print("***************************************************************")


D:\Anaconda3\envs\TF_36m\lib\site-packages\sklearn\linear_model\logistic.py:432: FutureWarning: Default solver will be changed to 'lbfgs' in 0.22. Specify a solver to silence this warning.
  FutureWarning)
D:\Anaconda3\envs\TF_36m\lib\site-packages\sklearn\utils\validation.py:752: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


(30704,)
(30704,)
***************************************************************


In [33]:
# fit a Naive Bayes model to the data
model = GaussianNB()
model.fit(traindata, trainlabel)
print(model)
# make predictions
expected = testlabel
predicted = model.predict(testdata)

# expected = expected.flatten()
#predicted = predicted.reshape(len(predicted),1)
print(expected.shape)
print(predicted.shape)

print(type(expected))

cm = ConfusionMatrix(expected, predicted)

expected = np.array(expected)
predicted = np.array(predicted)



cm.print_stats()

np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')

print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()
print("***************************************************************")

GaussianNB(priors=None, var_smoothing=1e-09)
(30704,)
(30704,)
<class 'numpy.ndarray'>
population: 30704
P: 15260
N: 15444
PositiveTest: 13923
NegativeTest: 16781
TP: 12819
TN: 14340
FP: 1104
FN: 2441
TPR: 0.84003931848
TNR: 0.928515928516
PPV: 0.920706744236
NPV: 0.85453787021
FPR: 0.0714840714841
FDR: 0.0792932557638
FNR: 0.15996068152
ACC: 0.884542730589
F1_score: 0.878525168763
MCC: 0.77189268434
informedness: 0.768555246996
markedness: 0.775244614447
prevalence: 0.497003647733
LRP: 11.7514195966
LRN: 0.172275646123
DOR: 68.2128894074
FOR: 0.14546212979


D:\Anaconda3\envs\TF_36m\lib\site-packages\sklearn\utils\validation.py:752: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Predicted  False   True  __all__
Actual                          
False      14340   1104    15444
True        2441  12819    15260
__all__    16781  13923    30704
(30704,)
(30704,)
***************************************************************


In [34]:




# fit a k-nearest neighbor model to the data
model = KNeighborsClassifier()
model.fit(traindata, trainlabel)
print(model)
# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model

#expected = expected.flatten()
print(expected.shape)
print(predicted.shape)

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()


# cm = metrics.confusion_matrix(expected, predicted)
# print(cm)
# tpr = float(cm[0][0])/np.sum(cm[0])
# fpr = float(cm[1][1])/np.sum(cm[1])
# print("%.3f" %tpr)
# print("%.3f" %fpr)
# print("Accuracy")
# print("%.3f" %ACC)
# print("precision")
# print("%.3f" %precision)
# print("recall")
# print("%.3f" %recall)
# print("f-score")
# print("%.3f" %f1)
# print("fpr")
# print("%.3f" %fpr)
# print("tpr")
# print("%.3f" %tpr)
print("***************************************************************")



model = DecisionTreeClassifier()
model.fit(traindata, trainlabel)
print(model)
# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model

#expected = expected.flatten()

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()
print("***************************************************************")

print("AdaBoostClassifier")
model = AdaBoostClassifier(n_estimators=100)
model.fit(traindata, trainlabel)

# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model

#expected = expected.flatten()

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()

print("***************************************************************")



print("RandomForestClassifier")
model = RandomForestClassifier(n_estimators=100)
model = model.fit(traindata, trainlabel)

# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model
#expected = expected.flatten()

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()

print("end****end***********************************************************")




D:\Anaconda3\envs\TF_36m\lib\site-packages\ipykernel_launcher.py:7: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  import sys


KNeighborsClassifier(algorithm='auto', leaf_size=30, metric='minkowski',
           metric_params=None, n_jobs=None, n_neighbors=5, p=2,
           weights='uniform')
(30704,)
(30704,)
population: 30704
P: 15260
N: 15444
PositiveTest: 15268
NegativeTest: 15436
TP: 15115
TN: 15291
FP: 153
FN: 145
TPR: 0.990498034076
TNR: 0.990093240093
PPV: 0.989979041132
NPV: 0.990606374708
FPR: 0.00990675990676
FDR: 0.0100209588682
FNR: 0.00950196592398
ACC: 0.990294424179
F1_score: 0.990238469602
MCC: 0.980588345
informedness: 0.980591274169
markedness: 0.98058541584
prevalence: 0.497003647733
LRP: 99.9820368514
LRN: 0.00959704151004
DOR: 10418.0060852
FOR: 0.00939362529153
Predicted  False   True  __all__
Actual                          
False      15291    153    15444
True         145  15115    15260
__all__    15436  15268    30704
(30704,)
(30704,)
***************************************************************
DecisionTreeClassifier(class_weight=None, criterion='gini', max_depth=None,
         

D:\Anaconda3\envs\TF_36m\lib\site-packages\sklearn\utils\validation.py:752: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


population: 30704
P: 15260
N: 15444
PositiveTest: 15184
NegativeTest: 15520
TP: 14796
TN: 15056
FP: 388
FN: 464
TPR: 0.969593709043
TNR: 0.974876974877
PPV: 0.974446786091
NPV: 0.970103092784
FPR: 0.025123025123
FDR: 0.0255532139094
FNR: 0.0304062909567
ACC: 0.972251172486
F1_score: 0.972014189988
MCC: 0.944510280567
informedness: 0.94447068392
markedness: 0.944549878874
prevalence: 0.497003647733
LRP: 38.5938279445
LRN: 0.0311898749692
DOR: 1237.38322076
FOR: 0.0298969072165
Predicted  False   True  __all__
Actual                          
False      15056    388    15444
True         464  14796    15260
__all__    15520  15184    30704
(30704,)
(30704,)
***************************************************************
RandomForestClassifier


D:\Anaconda3\envs\TF_36m\lib\site-packages\ipykernel_launcher.py:102: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().


population: 30704
P: 15260
N: 15444
PositiveTest: 15230
NegativeTest: 15474
TP: 15167
TN: 15381
FP: 63
FN: 93
TPR: 0.993905635649
TNR: 0.995920745921
PPV: 0.995863427446
NPV: 0.993989918573
FPR: 0.00407925407925
FDR: 0.00413657255417
FNR: 0.00609436435125
ACC: 0.994919228765
F1_score: 0.994883568383
MCC: 0.989839863702
informedness: 0.98982638157
markedness: 0.989853346019
prevalence: 0.497003647733
LRP: 243.648867253
LRN: 0.0061193266394
DOR: 39816.2872504
FOR: 0.00601008142691
Predicted  False   True  __all__
Actual                          
False      15381     63    15444
True          93  15167    15260
__all__    15474  15230    30704
(30704,)
(30704,)
end****end***********************************************************


In [35]:
model = svm.SVC(kernel='linear')#调参
model = model.fit(traindata, trainlabel)

# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model
expected = expected.flatten()
print(expected.shape)
print(predicted.shape)

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()

print("***************************************************************")

D:\Anaconda3\envs\TF_36m\lib\site-packages\sklearn\utils\validation.py:752: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


(30704,)
(30704,)
population: 30704
P: 15260
N: 15444
PositiveTest: 14575
NegativeTest: 16129
TP: 13929
TN: 14798
FP: 646
FN: 1331
TPR: 0.912778505898
TNR: 0.958171458171
PPV: 0.955677530017
NPV: 0.917477834956
FPR: 0.0418285418285
FDR: 0.0443224699828
FNR: 0.0872214941022
ACC: 0.93561099531
F1_score: 0.9337355455
MCC: 0.872051967345
informedness: 0.870949964069
markedness: 0.873155364973
prevalence: 0.497003647733
LRP: 21.8219059521
LRN: 0.0910291089955
DOR: 239.724481465
FOR: 0.0825221650443
Predicted  False   True  __all__
Actual                          
False      14798    646    15444
True        1331  13929    15260
__all__    16129  14575    30704
(30704,)
(30704,)
***************************************************************


In [36]:
from keras.models import Sequential, Model
from keras.layers import Dense, Dropout, Activation, Embedding
from keras.wrappers.scikit_learn import KerasClassifier
import h5py
from keras import callbacks
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
from keras.utils import to_categorical
def build_model():
    # 1. define the network
    model = Sequential()
    #model = Model()
    model.add(Dense(1024,input_dim=41,activation='relu'))  
    model.add(Dropout(0.01))
    model.add(Dense(1))
    model.add(Activation('sigmoid'))
    # try using different optimizers and different optimizer configs
    model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])   
   # model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])   
    return model

In [37]:
#DNN
checkpointer = callbacks.ModelCheckpoint(filepath="./DNNResult/checkpoint-{epoch:02d}.hdf5", verbose=1, save_best_only=True, monitor='loss')
#csv_logger = CSVLogger('./DNNResult/training_set_dnnanalysis.csv',separator=',', append=False)
model = KerasClassifier(build_fn=build_model, epochs=10, batch_size=84)
#model.fit(traindata, trainlabel, callbacks=[checkpointer,csv_logger])
model.fit(traindata, trainlabel, callbacks=[checkpointer])
#model.save("DNNResult/dnn1layer_model.hdf5")

# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model
expected = expected.flatten()
predicted = predicted.flatten()
print(predicted.shape)
print(expected.shape)

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()

print("***************************************************************")


Epoch 1/10
122813/122813 [==============================] - 5s 44us/step - loss: 0.1383 - acc: 0.9483

Epoch 00001: loss improved from inf to 0.13830, saving model to ./DNNResult/checkpoint-01.hdf5
Epoch 2/10
122813/122813 [==============================] - 5s 39us/step - loss: 0.0837 - acc: 0.9679

Epoch 00002: loss improved from 0.13830 to 0.08373, saving model to ./DNNResult/checkpoint-02.hdf5
Epoch 3/10
122813/122813 [==============================] - 5s 39us/step - loss: 0.0658 - acc: 0.9763

Epoch 00003: loss improved from 0.08373 to 0.06585, saving model to ./DNNResult/checkpoint-03.hdf5
Epoch 4/10
122813/122813 [==============================] - 5s 41us/step - loss: 0.0559 - acc: 0.9804

Epoch 00004: loss improved from 0.06585 to 0.05594, saving model to ./DNNResult/checkpoint-04.hdf5
Epoch 5/10
122813/122813 [==============================] - 5s 39us/step - loss: 0.0501 - acc: 0.9822

Epoch 00005: loss improved from 0.05594 to 0.05010, saving model to ./DNNResult/checkpoint-05